# llm-d Serving with Valkey: Routing, KV Cache and Batch Queue

> Put llm-d's router in front of two vLLM servers and use a single Valkey server for the router's prefix-cache index, the shared KV cache, and the batch job queue with per-user quotas.

**Difficulty:** Advanced · Python · ~30 min

**Who is this for:** Engineers serving LLMs on more than one vLLM instance who want to see what a cache-aware router does, and what Valkey stores for it.

**The scenario.** A contract-review assistant runs on two vLLM servers.
Reviewers chat with it about long lease documents, one question after another
about the same lease. An analyst runs batch questions over the same leases in
the background. Every section of this notebook follows that assistant.

**By the end you'll** see a reviewer's follow-up land on the server that
already holds the conversation, see a busy server hand a conversation to its
neighbor, see the routing index survive a router restart that an in-memory
index does not, and watch a per-user quota hold back the analyst's batch jobs.
All of that state lives in one Valkey server.

## The problem

A model server answers a request in two phases. First it reads the whole prompt
and computes the model's attention state, the KV cache, for every prompt
token. This is **prefill**. Then it generates the answer one token at a time.
The wait from sending a request to receiving the first generated token is the
**time to first token (TTFT)**. Most of it is prefill, and prefill grows with
the prompt. A reviewer's conversation starts at about a thousand tokens of
lease text, and every follow-up sends all of it again.

vLLM keeps the KV cache of recent prompts in memory (its prefix cache), so a
follow-up that reaches the same server skips most of the prefill. With one
server that happens by itself. With two servers and nothing coordinating them,
three things go wrong:

1. **Follow-ups land on a server that doesn't have the conversation**, which
   then prefills the whole conversation again. "Where the first token comes
   from", "Routing: follow-ups find their server" and "Baseline: random
   routing" show this.
2. **The server holding the conversation is busy while its neighbor is idle.**
   Sending the follow-up there anyway means it waits ("When the best server is
   busy").
3. **Batch work competes with chat for the same servers.** The analyst's
   questions can take the slots reviewers are waiting for ("Batch jobs share
   the same servers").

![Two vLLM servers, vllm-a and vllm-b, with no router in front of them. Chat requests are spread at random. A conversation's first turn runs on vllm-a, which now has its prefix cached; the follow-up lands on vllm-b, which has to recompute the whole prompt or reload it from a shared cache.](images/01-without-coordination.svg)

The [KV caching recipe](../kv-caching/) solves part of the first problem: two
vLLM servers share one KV cache in Valkey, so the second server loads the
conversation instead of recomputing it. A shared KV cache removes the
recompute but not the placement problem. Loading from Valkey is still slower
than finding the prefix in the server's own memory, and nothing decides which
server gets each request.

## The solution at a glance

[llm-d](https://llm-d.ai) is an open-source, Kubernetes-native stack for
distributed LLM inference that runs model servers such as vLLM and SGLang. This
recipe uses two of its parts, with upstream vLLM as the model server, and runs
them without Kubernetes:

- **The router**, Envoy plus llm-d's Endpoint Picker (EPP). It listens to each
  vLLM server's KV-cache events, keeps an index of which server holds which
  prompt blocks, and sends each request to the server with the longest
  matching prefix, unless that server is too busy. On Kubernetes the EPP finds
  the vLLM servers by watching an `InferencePool`, which selects the
  model-server pods by label, so it follows servers as they are added, removed
  or rescheduled. Without Kubernetes there is nothing to watch, so this recipe
  uses llm-d's
  [no-Kubernetes setup](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/no-kubernetes-deployment):
  the EPP reads a fixed list of servers from
  [`epp/endpoints.yaml`](epp/endpoints.yaml). Adding a server means editing
  that file and restarting the router.
- **The batch processor**, llm-d-async. It holds background jobs in a queue,
  applies per-user quotas, and sends admitted jobs through the same router.

![Architecture of the stack. A chat client sends requests to Envoy on localhost:8080, which asks the EPP (Endpoint Picker) which vLLM replica to use and forwards the request to vllm-a or vllm-b. Both replicas publish KV events over ZMQ to the EPP, which keeps its prefix-cache index in Valkey database 1. The replicas exchange KV chunks with the LMCache server, which stores them in Valkey database 0. Batch jobs submitted from Python go into a Valkey sorted set with per-user quota keys; llm-d-async admits them, sends them through Envoy like chat requests, and writes results to a Valkey list. A legend colours the three Valkey roles: index blue, KV cache green, queue and quotas orange.](images/02-architecture.svg)

| Component | Role | Valkey use |
| --- | --- | --- |
| Envoy | Front door on `localhost:8080`; asks the EPP where each request goes (ext_proc) and forwards it | none |
| llm-d EPP (Endpoint Picker) v1.0.0 | Tokenizes each request, scores replicas by cached prefix and load, picks one | prefix-cache index (db 1): `<hash>` hashes and `engine:<hash>` sorted sets |
| vLLM 0.28 (x2, CPU) | Serves the model; keeps its own prefix cache; publishes KV events over ZMQ | none directly |
| LMCache 0.5.4 server | Takes KV from both replicas and serves it back | KV chunks (db 0): one string per 256 tokens |
| llm-d-async v1.0.0 | Pulls batch jobs, applies per-user quotas, sends jobs through Envoy | `request-sortedset`, `result-list`, `quota:*`, `request-*` (db 0) |
| Valkey 9.1 (valkey-bundle) | One server for all of the above | |

In this stack a single Valkey server has three jobs. The colours match the
diagram and the key tables later on:

| | Job | State in Valkey | Who writes and reads it | What breaks without Valkey |
| --- | --- | --- | --- | --- |
| 🟦 | Prefix-cache index | Which replica holds each 64-token prompt block (database 1) | The router writes it from the replicas' KV events and reads it for every request | The index lives in the router's memory. A restarted router starts empty and rebuilds only from the replicas' recent events ("The index outlives the router") |
| 🟩 | Shared KV cache | The computed KV, in 256-token chunks (database 0) | LMCache writes it for either replica and reads it back for either | A replica that doesn't have the prefix in memory recomputes it ("Where the first token comes from", "When the best server is busy") |
| 🟧 | Batch queue and per-user quotas | Queued jobs, results, and each user's recent admissions (database 0) | Python submits jobs; llm-d-async takes them, checks and updates the quota, and writes results | Queue and quota counts are one processor's in-memory state: lost when it restarts, and not shared with a second processor |

None of the llm-d components is modified: each is the upstream image, pointed at
Valkey by configuration.

**Why CPU here?** This recipe runs on CPU so readers can try it locally without
GPU hardware. llm-d is validated in production on GPUs and TPUs and also
supports CPU; its
[upstream routing guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/precise-prefix-cache-routing)
defaults to NVIDIA GPUs. The three Valkey roles are useful in either setup.
This Compose stack and the timings shown here were tested only on CPU. A GPU
version would need an appropriate vLLM image, GPU device configuration, and
router load settings calibrated for that hardware.

## How to run this notebook

The [recipe README](README.md) has the prerequisites (a container runtime with
10 GiB of memory, or at least 8.5 GiB, and [uv](https://docs.astral.sh/uv/)) and the one-time
setup. In short, from this directory:

```bash
./start.sh
uv sync
uv run python -m ipykernel install --user --name llm-d-serving-valkey
uv run jupyter lab llm-d-serving-with-valkey.ipynb
```

> ⚠ **Security:** These examples use no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

Select the `llm-d-serving-valkey` kernel and run the cells top to bottom. Later
cells build on earlier ones. A few cells restart or reconfigure the router with
`docker compose` (or `podman compose`; the notebook uses whichever is
installed), so run the notebook from this directory. The cells took about
three minutes in total in our runs on an 8-CPU, 16 GiB Colima VM on an Apple M4
Pro.

## Connect to the stack

The setup cell first checks that the container runtime has at least the 8.5 GiB
of memory this stack needs (10 GiB is recommended), and stops with a message if
it doesn't. Then it defines the clients and helpers that every later cell uses. Helpers
that the recipe's tests also use live in [`helpers.py`](helpers.py), so the
tests exercise the same code; the rest are defined in the cell itself:

- `router` is an OpenAI-compatible client for Envoy on `localhost:8080`, so the
  EPP picks the replica for every request it sends. `direct` is a dictionary
  of two more clients, keyed by replica name: `direct["vllm-a"]` talks to
  vllm-a on `localhost:8001` and `direct["vllm-b"]` to vllm-b on
  `localhost:8002`, skipping the router. Later cells use them to send a request
  to a chosen replica, for example to compare recompute, Valkey load and local
  hit on the same prompt.
- `valkey` reads database 0 (LMCache's KV chunks and llm-d-async's queue), and
  `router_index` reads database 1 (the router's index).
- `ask()` sends one streaming chat request and reports its TTFT, which replica
  served it, and how many prompt tokens came from that replica's memory or
  from Valkey.
- `settled_dbsize()` waits until database 0 stops growing, so that a replica
  only looks for a prefix in Valkey after the writes have landed.

The "Helper notes" after the cell say how `ask()` and `settled_dbsize()` work.

In [ ]:
import http.client
import json
import os
import shutil
import statistics
import subprocess
import time
import urllib.request
import uuid

import yaml
from openai import OpenAI

from helpers import (close_connections, connect, document, metric, scrape,
                     settled_dbsize, submit_job)

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ROUTER_URL = "http://127.0.0.1:8080"  # Envoy; the EPP picks the replica
REPLICA_URLS = {"vllm-a": "http://127.0.0.1:8001", "vllm-b": "http://127.0.0.1:8002"}
# The router names replicas by "<ip>:<port>"; epp/endpoints.yaml maps those to names.
with open("epp/endpoints.yaml") as f:
    POD_NAMES = {f"{e['address']}:{e['port']}": e["name"] for e in yaml.safe_load(f)["endpoints"]}
EPP_METRICS = "http://127.0.0.1:9090/metrics"
ASYNC_METRICS = "http://127.0.0.1:9091/metrics"
VALKEY_HOST = os.environ.get("VALKEY_HOST", "127.0.0.1")
VALKEY_PORT = int(os.environ.get("VALKEY_PORT", "6379"))

router = OpenAI(base_url=f"{ROUTER_URL}/v1", api_key="not-needed", max_retries=0)
direct = {name: OpenAI(base_url=f"{url}/v1", api_key="not-needed", max_retries=0)
          for name, url in REPLICA_URLS.items()}


valkey = connect(VALKEY_HOST, VALKEY_PORT, 0)        # LMCache KV chunks + llm-d-async queue and quotas
router_index = connect(VALKEY_HOST, VALKEY_PORT, 1)  # the router's prefix-cache index


def detect_engine():
    """First of docker/podman that is installed and answers `info`."""
    forced = os.environ.get("CONTAINER_ENGINE")
    for candidate in [forced] if forced else ["docker", "podman"]:
        if candidate and shutil.which(candidate):
            if subprocess.run([candidate, "info"], capture_output=True, timeout=30).returncode == 0:
                return candidate
    raise RuntimeError("No working container runtime found (tried docker, podman).")


ENGINE = detect_engine()

# Preflight: stop now if the runtime has less memory than the stack needs,
# instead of a replica dying halfway through the notebook. The thresholds match
# start.sh. A runtime reports a little less than its setting
# (the VM keeps some): the measured minimum of 8.5 GiB reports about 8.2 GiB,
# and the recommended 10 GiB about 9.7 GiB.
MIN_RUNTIME_MEMORY = 8_589_934_592            # 8.0 GiB reported
RECOMMENDED_RUNTIME_MEMORY = 10_200_547_328   # 9.5 GiB reported


def runtime_memory_bytes():
    """Memory the container runtime reports (Docker, then Podman's field), or None."""
    for fmt in ("{{.MemTotal}}", "{{.Host.MemTotal}}"):
        out = subprocess.run([ENGINE, "info", "--format", fmt], capture_output=True,
                             text=True, timeout=30).stdout.strip()
        if out.isdigit():
            return int(out)
    return None


RUNTIME_MEMORY = runtime_memory_bytes()
if RUNTIME_MEMORY is not None and RUNTIME_MEMORY < MIN_RUNTIME_MEMORY:
    raise SystemExit(
        f"{ENGINE} reports {RUNTIME_MEMORY / 2**30:.1f} GiB of memory; this recipe needs at least "
        "8.5 GiB (10 GiB recommended). Raise the runtime's memory limit (README, 'Memory "
        "requirements'), restart the stack, and run this cell again.")
if RUNTIME_MEMORY is not None and RUNTIME_MEMORY < RECOMMENDED_RUNTIME_MEMORY:
    print(f"Note: {ENGINE} reports {RUNTIME_MEMORY / 2**30:.1f} GiB, below the recommended 10 GiB. "
          "The notebook should run, but restart the stack (docker compose down, then up) before "
          "running it again (README, 'Running on a smaller machine').")


def compose(*args, **env):
    """Run `<engine> compose ...` in this directory, with extra env vars."""
    result = subprocess.run([ENGINE, "compose", *args], capture_output=True, text=True,
                            env={**os.environ, **env}, timeout=600)
    if result.returncode != 0:
        raise RuntimeError(f"{ENGINE} compose {' '.join(args)} failed:\n{result.stderr}")


def replica_counters():
    counters = {}
    for name, url in REPLICA_URLS.items():
        text = scrape(f"{url}/metrics")
        counters[name] = {
            "served": metric(text, "vllm:request_success_total"),
            "local": metric(text, "vllm:prefix_cache_hits_total"),
            "valkey": metric(text, "vllm:external_prefix_cache_hits_total"),
        }
    return counters


def served_by(before, timeout_s=3.0):
    """Which replica finished a request since `before`, plus its cache-hit deltas."""
    deadline = time.monotonic() + timeout_s
    while True:
        after = replica_counters()
        moved = [n for n in after if after[n]["served"] > before[n]["served"]]
        if moved or time.monotonic() > deadline:
            break
        time.sleep(0.05)
    name = moved[0] if len(moved) == 1 else "?"
    if name == "?":
        return name, 0, 0
    return (name, int(after[name]["local"] - before[name]["local"]),
            int(after[name]["valkey"] - before[name]["valkey"]))


def ask(client, messages, max_tokens=16):
    """One streaming chat request; returns timing, cache and placement signals."""
    before = replica_counters()
    start = time.perf_counter()
    ttft, text, usage = None, "", None
    stream = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0,
        stream=True, stream_options={"include_usage": True})
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            ttft = ttft or (time.perf_counter() - start)
            text += chunk.choices[0].delta.content
        if chunk.usage:
            usage = chunk.usage
    replica, local, from_valkey = served_by(before)
    details = usage.prompt_tokens_details
    return {"replica": replica, "ttft_ms": round((ttft or time.perf_counter() - start) * 1000),
            "prompt": usage.prompt_tokens, "cached": details.cached_tokens if details else 0,
            "local": local, "valkey": from_valkey, "text": text}


memory = f"{RUNTIME_MEMORY / 2**30:.1f} GiB" if RUNTIME_MEMORY else "not reported"
print(f"container runtime: {ENGINE}, memory {memory}")
print("Setup ready.")

### Helper notes

- **How `ask()` finds the replica.** vLLM's responses do not name the replica,
  so `ask()` reads each replica's `vllm:request_success_total` counter before
  and after the request and sees which one went up.
- **Where the cached tokens came from.** `vllm:prefix_cache_hits_total` counts
  tokens found in the replica's own memory, and
  `vllm:external_prefix_cache_hits_total` counts tokens LMCache loaded from
  Valkey. The `cached` column in later tables is what vLLM reports to the
  client in `usage.prompt_tokens_details.cached_tokens`; it counts both kinds.
- **Why `settled_dbsize()` exists.** The LMCache server writes KV to Valkey
  asynchronously. Before a cell sends a prompt to the other replica and expects
  it to load the prefix from Valkey, it waits until database 0 has not changed
  for one second.

Check that everything is up before going further: Valkey answers, both replicas
are healthy, and a request through the router succeeds. The cell also makes sure
the router runs its default configuration (`epp/precise-valkey.yaml`), in case
an earlier run stopped halfway through a section that switches it. Envoy and the
EPP images have no shell for a container healthcheck, so `docker compose up
--wait` cannot tell when the router is ready; `wait_for_router()` retries a
one-token request until one goes through.

In [ ]:
def wait_for_router(timeout_s=120):
    deadline = time.monotonic() + timeout_s
    while True:
        try:
            router.completions.create(model=MODEL, prompt="ping", max_tokens=1, timeout=10)
            return
        except Exception as err:  # connection refused / 503 while the EPP starts
            if time.monotonic() > deadline:
                raise RuntimeError("The router did not answer; check `docker compose logs epp`.") from err
            time.sleep(1)


def use_routing(strategy):
    """Recreate the EPP with epp/<strategy>.yaml (no-op if it already runs it)."""
    compose("up", "-d", "epp", EPP_STRATEGY=strategy)
    wait_for_router()


assert valkey.ping() in (b"PONG", "PONG"), "Valkey is not answering"
for name, url in REPLICA_URLS.items():
    with urllib.request.urlopen(f"{url}/health", timeout=5) as resp:
        assert resp.status == 200, f"{name} is not healthy"
use_routing("precise-valkey")
print("valkey      : PONG")
print("replicas    :", ", ".join(REPLICA_URLS))
print("router      : answering on", ROUTER_URL, "(precise-valkey)")

## What Valkey holds, before anything runs

Before the first reviewer opens a lease, look at what Valkey holds. Three kinds
of keys will show up as the notebook runs, coloured as in the diagram:

| | Database | Keys | Written by | What it is |
| --- | --- | --- | --- | --- |
| 🟦 | 1 | `engine:<hash>` (sorted set) | router | maps a vLLM block hash to the router's own hash for the same block |
| 🟦 | 1 | `<hash>` (hash) | router | one key per cached 64-token block; each field names a replica that holds it |
| 🟩 | 0 | `Qwen/Qwen2.5-0.5B-Instruct@...` (string) | LMCache server | one 256-token KV chunk, about 3 MiB |
| 🟧 | 0 | `request-sortedset`, `result-list`, `quota:*`, `request-*` | llm-d-async | batch job queue, results, per-user quotas, per-job state |

The router index has a database of its own, so that clearing a restarted
replica's entries never runs into LMCache's keys (see "Notes on the router
index").

`key_census()` counts them. Right after startup the router index already has a
few entries: each replica's healthcheck warm-up filled one 64-token block. There
are no LMCache chunks yet, because the warm-up is shorter than one 256-token
chunk. (On a re-run you will also see what earlier runs left behind; nothing in
the notebook depends on an empty Valkey.)

In [ ]:
QUEUE_KEYS = ("request-sortedset", "result-list")
QUEUE_PREFIXES = ("quota:", "request-seq:", "request-active:", "request-cancel:", "request-payload:")


def scan_keys(client):
    cursor, keys = b"0", []
    while True:
        cursor, batch = client.scan(cursor, count=1000)
        keys += [k.decode() for k in batch]
        if cursor in (b"0", "0"):
            return keys


def key_census():
    census = {"router engine:<hash>": 0, "router <hash>": 0, "LMCache chunks": 0,
              "batch queue": 0, "other": 0}
    for key in scan_keys(router_index):
        census["router engine:<hash>" if key.startswith("engine:") else
               "router <hash>" if key.isdigit() else "other"] += 1
    for key in scan_keys(valkey):
        census["LMCache chunks" if key.startswith(f"{MODEL}@") else
               "batch queue" if key in QUEUE_KEYS or key.startswith(QUEUE_PREFIXES) else
               "other"] += 1
    return census


def show_census(title):
    print(f"{title}: DBSIZE db0={valkey.custom_command(['DBSIZE'])} "
          f"db1={router_index.custom_command(['DBSIZE'])}")
    for kind, count in key_census().items():
        print(f"  {kind:22s} {count:5d}")


show_census("Valkey now")

## Where the first token comes from: recompute, Valkey load, local hit

Every reviewer request lands in one of three cases, depending on what the
server it reaches already has for that conversation. Where the router sends a
request decides which case it lands in, so this section measures all three
directly, without the router. The prompt is a reviewer's first question about a
29-clause lease: roughly 1,000 to 1,150 tokens including the system
instruction, depending on the random tag repeated in each clause.

- **Recompute**: no replica has seen it. vLLM prefills every token.
- **Load from Valkey**: another replica computed it, so the KV is in Valkey but
  not in this replica's memory. LMCache loads it from Valkey in 256-token
  chunks and vLLM prefills only the rest.
- **Local hit**: this replica has it in its own memory (vLLM's prefix cache).

The cell sends the same fresh prompt to replica A (recompute), then to replica B
twice (load from Valkey, then local hit), directly rather than through the
router, and repeats with three different leases. Before B's first request the
cell waits for A's KV to reach Valkey (`settled_dbsize()`). `max_tokens=1`, so
the timing is prefill plus one token. A fresh `uuid` in each lease keeps the
first request cold on every run.

In [ ]:
QUESTIONS = ["Summarise clause 3.", "What does clause 9 require?", "And clause 14?"]


def new_conversation(label):
    tag = f"{label}-{uuid.uuid4().hex[:8]}"
    instruction = "You are a contract-review assistant. Answer the reviewer's questions about this lease."
    return [{"role": "system", "content": f"{instruction}\n\n{document(tag)}"}]


cases = {"recompute": [], "load from Valkey": [], "local hit": []}
for _ in range(3):
    prompt = new_conversation("latency") + [{"role": "user", "content": QUESTIONS[0]}]
    cases["recompute"].append(ask(direct["vllm-a"], prompt, max_tokens=1))
    settled_dbsize(valkey)  # LMCache stores A's KV in Valkey asynchronously
    cases["load from Valkey"].append(ask(direct["vllm-b"], prompt, max_tokens=1))
    cases["local hit"].append(ask(direct["vllm-b"], prompt, max_tokens=1))

print(f"{'case':18s} {'TTFT ms (3 runs)':22s} {'median':>7s} {'cached':>7s} {'local':>6s} {'Valkey':>7s}")
for case, runs in cases.items():
    last = runs[-1]
    print(f"{case:18s} {str([r['ttft_ms'] for r in runs]):22s} "
          f"{statistics.median(r['ttft_ms'] for r in runs):7.0f} {last['cached']:7d} "
          f"{last['local']:6d} {last['valkey']:7d}   (of {last['prompt']} prompt tokens)")

prefill_rate = statistics.median(r["prompt"] / (r["ttft_ms"] / 1000) for r in cases["recompute"])
print(f"\nmeasured prefill rate on this machine: {prefill_rate:,.0f} tokens/s "
      "(epp/precise-valkey.yaml assumes 1,000)")

The local hit skips nearly all of the prefill. Loading from Valkey is slower
than a local hit, because about 3 MiB per 256-token chunk travels from Valkey to
the LMCache server to vLLM, and LMCache only stores whole chunks, so the last
partial chunk is recomputed (the `Valkey` column stops at a multiple of 256). It
is still several times faster than recomputing. Across our runs on an 8-CPU,
16 GiB Colima VM on an Apple M4 Pro, the medians were about 1.1 to 1.35 s
to recompute, 0.15 to 0.35 s to load from Valkey, and 30 to 80 ms for a local
hit. Your numbers depend on the CPU.

The last line is the prefill rate that the recompute case implies. "When the
best server is busy" uses it.

So the router's goal is to make most requests local hits, and Valkey makes the
misses cheap. Next, the reviewers' conversations go through the router.

## Routing: follow-ups find their server

Now the reviewers' requests go through the router with its default
configuration ([`epp/precise-valkey.yaml`](epp/precise-valkey.yaml)). The
pieces that matter:

```yaml
- type: precise-prefix-cache-producer
  parameters:
    tokenProcessorConfig:
      blockSizeTokens: 64              # = vLLM --block-size
    indexerConfig:
      kvBlockIndexConfig:
        redisConfig:
          address: redis://valkey:6379/1   # the index lives in Valkey, database 1
    kvEventsConfig:
      discoverPods: true               # subscribe to every replica's KV events
- type: prefix-cache-affinity-filter   # keep only replicas holding >= 80% of the prompt
- type: token-load-scorer              # among those, pick the least-loaded
```

For every request the router tokenizes the prompt (by asking vLLM's
`/v1/chat/completions/render` endpoint), hashes it into 64-token blocks the same
way vLLM does, and looks the block hashes up in Valkey. Each replica publishes a
*block stored* or *block removed* event whenever its prefix cache changes; the
router subscribes to both replicas and keeps the index in Valkey up to date.

![One chat request through the router. Envoy passes it to the EPP, which (1) tokenizes the prompt through a replica's render endpoint, (2) hashes it into 64-token blocks, (3) looks the block hashes up in Valkey database 1, (4) scores the replicas by longest cached prefix and in-flight load, and (5) picks one; Envoy forwards the request to that vLLM replica.](images/03-request-path.svg)

Four reviewers each ask three questions about their own lease. Turn 1 is cold
everywhere, so the router places it by load (both replicas are idle, so the
choice is effectively random). Turns 2 and 3 resend the whole conversation,
and the router should send them to the replica that served turn 1.

In [ ]:
def run_conversations(client, label, count, turns=3):
    """Interleave `count` conversations turn by turn; return one row per request."""
    conversations = [new_conversation(f"{label}{i}") for i in range(count)]
    rows = []
    for turn in range(turns):
        for i, messages in enumerate(conversations):
            messages.append({"role": "user", "content": QUESTIONS[turn]})
            result = ask(client, messages)
            messages.append({"role": "assistant", "content": result["text"]})
            rows.append({"conversation": i, "turn": turn + 1, **result})
    return conversations, rows


def print_rows(rows):
    print("conv turn  replica  TTFT ms  cached  local  Valkey")
    for r in rows:
        print(f"{r['conversation']:4d} {r['turn']:4d}   {r['replica']:7s} {r['ttft_ms']:8d} "
              f"{r['cached']:7d} {r['local']:6d} {r['valkey']:7d}")


def follow_up_summary(rows):
    first = {r["conversation"]: r["replica"] for r in rows if r["turn"] == 1}
    follow = [r for r in rows if r["turn"] > 1]
    same = [r for r in follow if r["replica"] == first[r["conversation"]]]
    moved = [r for r in follow if r["replica"] != first[r["conversation"]]]
    print(f"\nfollow-ups on the replica that served turn 1: {len(same)}/{len(follow)}")
    for label, group in (("stayed", same), ("moved", moved)):
        if group:
            print(f"  {label:6s}: median TTFT {statistics.median(r['ttft_ms'] for r in group):5.0f} ms, "
                  f"tokens from local memory {sum(r['local'] for r in group):6d}, "
                  f"from Valkey {sum(r['valkey'] for r in group):6d}")
    return same, moved


precise_conversations, precise_rows = run_conversations(router, "precise", count=4)
print_rows(precise_rows)
follow_up_summary(precise_rows);

### What the router stores in Valkey

The router's index is in database 1. Each 64-token block of every prompt a
replica has cached is one Valkey hash, keyed by the router's hash of the
block's tokens (and of every block before it, so a hash identifies a whole
prefix). Each field in that hash is a small JSON record naming a replica that
holds the block. Fields with `"Speculative":true` are the router's own guess:
with `speculativeIndexing: true` it records a routed request's blocks against
the chosen replica for 2 seconds, before the replica's events confirm them.
`engine:<hash>` sorted sets map vLLM's own block hashes, which
arrive in the events, to the router's hashes. The cell below shows one entry and
counts the confirmed (non-speculative) blocks the index assigns to each
replica. (What can go stale in this index, and why it has a database of its
own, is in "Notes on the router index".)

In [ ]:
def confirmed_entries(key):
    """The block's fields that a replica reported (not the router's 2 s guesses)."""
    return [e for e in map(json.loads, router_index.hkeys(key)) if not e.get("Speculative")]


def index_by_replica():
    """Confirmed blocks per replica."""
    counts = {name: 0 for name in POD_NAMES.values()}
    for key in scan_keys(router_index):
        if key.isdigit():
            for entry in confirmed_entries(key):
                # An address not in epp/endpoints.yaml (say, left over from
                # before a subnet change) is counted under its own name.
                name = POD_NAMES.get(entry["PodIdentifier"], entry["PodIdentifier"])
                counts[name] = counts.get(name, 0) + 1
    return counts


# Show a block both replicas reported, such as the healthcheck warm-up block.
block_key = max((k for k in scan_keys(router_index) if k.isdigit()),
                key=lambda k: len({e["PodIdentifier"] for e in confirmed_entries(k)}))
engine_key = next(k for k in scan_keys(router_index) if k.startswith("engine:"))
print(f"{block_key} ({router_index.type(block_key).decode()}):")
for entry in confirmed_entries(block_key):
    print("   field:", json.dumps(entry, separators=(",", ":")))
print(f"{engine_key} ({router_index.type(engine_key).decode()}):",
      router_index.custom_command(["ZRANGE", engine_key, "0", "-1"]))
lmcache_key = next(k for k in scan_keys(valkey) if k.startswith(f"{MODEL}@"))
print(f"\nLMCache chunk {lmcache_key[:60]}... ({valkey.type(lmcache_key).decode()}, "
      f"{valkey.strlen(lmcache_key):,} bytes)")
print("\nblocks the router index assigns to each replica:", index_by_replica())
show_census("\nValkey now")

The block printed first is one that both replicas have cached, so its hash has
two fields, one per replica. In each field, `PodIdentifier` is the replica's
address and port from `epp/endpoints.yaml`. `DeviceTier` says `gpu` even
though these replicas run on CPU, because vLLM labels everything in its own
prefix cache `gpu`, whatever hardware it runs on.

## Baseline: random routing

What would the reviewers get without the index? For a baseline, switch the
router to [`epp/random.yaml`](epp/random.yaml): the same Envoy and EPP, but the
EPP picks a replica at random and does not look at the index. The cell
recreates the EPP container with `EPP_STRATEGY=random docker compose up -d epp`.

Five reviewers ask three questions each. The first question starts the
conversation and the other two are follow-ups, so there are ten follow-ups. With two
replicas, a follow-up has a coin-flip chance of landing on the replica that
does not have the conversation in memory. Without a shared cache that replica
would recompute the whole conversation. Here it loads the conversation from
Valkey, which the `Valkey` column shows.

In [ ]:
use_routing("random")
random_conversations, random_rows = run_conversations(router, "random", count=5)
print_rows(random_rows)
stayed, moved = follow_up_summary(random_rows)
use_routing("precise-valkey")
print("\nrouter switched back to precise-valkey")


def strategy_row(name, rows):
    """One summary line for a strategy's follow-up turns (turn 2 onwards)."""
    first = {r["conversation"]: r["replica"] for r in rows if r["turn"] == 1}
    follow = [r for r in rows if r["turn"] > 1]
    moved = sum(r["replica"] != first[r["conversation"]] for r in follow)
    local_hits = sum(r["valkey"] == 0 and r["local"] > 0 for r in follow)
    median = statistics.median(r["ttft_ms"] for r in follow)
    print(f"{name:15s} {len(follow):10d} {moved:6d} {local_hits:11d} "
          f"{sum(r['valkey'] for r in follow):14d} {median:15.0f}")


print("\nfollow-up turns side by side")
print("strategy         follow-ups  moved  local hits  Valkey tokens  median TTFT ms")
strategy_row("precise-valkey", precise_rows)
strategy_row("random", random_rows)

With random routing, between 3 and 8 of the ten follow-ups moved to the replica
that did not have the conversation in our runs (your count will differ,
because the picks are random). Each of those loaded the conversation from
Valkey; the `Valkey tokens` column is the work that random routing cost and
precise routing avoided, since precise routing kept every follow-up on its
replica as a local hit. Without the shared cache, each moved follow-up would
have recomputed the whole conversation, which took 1.1 to 1.4 s for a first turn
in our runs. A follow-up that moves to a replica that already has the
conversation from an earlier moved turn is local again.

On this stack routing shows up as work avoided more than as lower latency,
because loading a conversation from Valkey is already cheap. In our runs the
median follow-up TTFT was 100 to 118 ms with precise routing and 102 to 160 ms
with random routing, against more than a second to recompute. The router and the
shared cache do different jobs. The router sends each follow-up to the replica
that already has the conversation in memory, so most requests are local hits.
When a request goes to the other replica anyway, LMCache loads the conversation
from Valkey instead of recomputing it.

The router keeps a reviewer on the same server as long as that server has room.
The next section fills that server.

## When the best server is busy

When the replica that holds a reviewer's conversation is full, the router
sends the next question to the idle replica, and that replica
loads most of the conversation from Valkey instead of recomputing it.

In the scenario, the analyst has started two long reports on a standard lease
template, and both run on replica A, which also holds a reviewer's
conversation. Each replica runs at most two requests at a time
(`--max-num-seqs 2` in `docker-compose.yml`), so A is full.

![Timeline of the busy-server example. vllm-a holds the conversation's prefix but is running two long jobs, so it has many tokens in flight; vllm-b is idle. The reviewer's next turn arrives at the router, which sees vllm-a's in-flight load, overrides affinity (load_override) and sends the turn to vllm-b. vllm-b loads most of the prefix from Valkey through LMCache and computes only the rest.](images/04-busy-server.svg)

The example:

1. Warm a "report template" prefix and a conversation's first turn on replica
   A, directly, so the router's index says A has both, and wait for their KV
   to reach Valkey.
2. Through the router, start two long report requests on the template
   (1,000 output tokens each, `ignore_eos`). Both go to A, which has the
   template; when the second one is routed, A has about 1,000 tokens in flight,
   under the router's limit of 1,500 (explained after the result). A is now
   full.
3. Send the conversation's next turn through the router. A has about 2,000
   tokens in flight and B none, so the router breaks stickiness and sends the
   conversation to B, which loads it from Valkey.
4. Close the two long requests (vLLM aborts them) and repeat with fresh
   prompts.

The decision depends only on token counts the router itself tracks, so it is
the same on any hardware. The first run also measures the alternatives: the same
conversation sent to A anyway (it waits for a free slot) and a same-sized
conversation that B has to recompute.

In [ ]:
def router_inflight_tokens():
    text = scrape(EPP_METRICS)
    return {name: int(metric(text, "llm_d_epp_inflight_tokens", endpoint_name=name))
            for name in REPLICA_URLS}


def affinity_decisions():
    text = scrape(EPP_METRICS)
    return {outcome: metric(text, "llm_d_epp_prefix_cache_affinity_filter_decisions_total", outcome=outcome)
            for outcome in ("sticky", "load_override", "no_match")}


def start_long_request(prompt, connections, max_tokens=1000):
    """Non-streaming completion through the router, left open on `connections`."""
    conn = http.client.HTTPConnection("127.0.0.1", 8080, timeout=600)
    connections.append(conn)  # tracked first, so a failed send is closed too
    body = {"model": MODEL, "prompt": prompt, "max_tokens": max_tokens,
            "ignore_eos": True, "temperature": 0}
    conn.request("POST", "/v1/completions", json.dumps(body), {"Content-Type": "application/json"})


def busy_replica_run(measure_alternatives):
    tag = uuid.uuid4().hex[:8]
    template = f"[report template {tag}] " + document(f"template-{tag}", clauses=26)
    direct["vllm-a"].completions.create(model=MODEL, prompt=template, max_tokens=1)
    conversation = new_conversation("busy")
    conversation.append({"role": "user", "content": QUESTIONS[0]})
    conversation.append({"role": "assistant", "content": ask(direct["vllm-a"], conversation)["text"]})
    conversation.append({"role": "user", "content": QUESTIONS[1]})
    settled_dbsize(valkey)  # the conversation's KV is in Valkey before B might need it

    decisions_before = affinity_decisions()
    long_requests = []
    try:  # if anything below fails, still abort the long requests
        for job in range(2):
            start_long_request(f"{template}\nReport {job}: write it in full.", long_requests)
            time.sleep(0.3)  # let the router see the first before routing the second
        time.sleep(0.5)      # demo pacing only: both requests are now decoding on a replica
        inflight = router_inflight_tokens()
        running = {n: int(metric(scrape(f"{u}/metrics"), "vllm:num_requests_running"))
                   for n, u in REPLICA_URLS.items()}
        routed = ask(router, conversation)
        alternatives = {}
        if measure_alternatives:
            same_size = new_conversation("busy-cold") + conversation[1:]
            alternatives["B recomputes a same-sized conversation"] = ask(direct["vllm-b"], same_size)
            alternatives["A anyway (waits for a free slot)"] = ask(direct["vllm-a"], conversation)
    finally:
        close_connections(long_requests)
    decisions = {k: int(v - decisions_before[k]) for k, v in affinity_decisions().items()}
    return inflight, running, routed, alternatives, decisions


results = []
for run in range(3):
    inflight, running, routed, alternatives, decisions = busy_replica_run(measure_alternatives=(run == 0))
    results.append(routed["replica"])
    print(f"run {run + 1}: before the turn, router in-flight tokens {inflight}, "
          f"vLLM running requests {running}")
    print(f"       affinity filter decisions {decisions}")
    print(f"       conversation -> {routed['replica']}: TTFT {routed['ttft_ms']} ms, "
          f"{routed['valkey']} tokens loaded from Valkey, {routed['local']} local "
          f"(of {routed['prompt']})")
    for label, alt in alternatives.items():
        print(f"       for comparison, {label}: TTFT {alt['ttft_ms']} ms "
              f"({alt['local']} local, {alt['valkey']} from Valkey)")
time.sleep(1)
print(f"\nconversation moved to the idle replica in {results.count('vllm-b')}/3 runs; "
      f"router in-flight tokens after aborting the long requests: {router_inflight_tokens()}")

In each run you should see the two long requests stay on A (`sticky` 2) and the
conversation go to B (`load_override` 1). B has never seen the conversation,
but most of it comes from Valkey, so its first token arrives in a fraction of
the time B needs to recompute a conversation of the same size: in our runs on
an 8-CPU, 16 GiB Colima VM on an Apple M4 Pro, 0.4 to 0.65 s against 2.2 to
2.4 s. Sending it to A anyway would be a local hit, but only after A finishes
one of its long jobs, which took 16 to 17 s.

On Podman for Windows, the cell above once failed with Envoy's `upstream
connect error ... connection termination` (one of three notebook runs; nothing
restarted and the logs showed nothing). Running the notebook again on the
same stack passed, so if you see it, re-run this section.

### Why the hand-off happens at CPU scale

The `prefix-cache-affinity-filter` estimates each replica's wait from the tokens
the router has in flight to it, divided by `peakPrefillThroughput`, and gives up
on the sticky replica when its estimated wait is more than `maxTTFTPenaltyMs`
longer than another replica's. llm-d's defaults are calibrated for GPUs, so this
recipe changes three router settings, all in
[`epp/precise-valkey.yaml`](epp/precise-valkey.yaml):

| Setting | Default | Here | Why |
| --- | --- | --- | --- |
| `inflight-load-producer` `addEstimatedOutputTokens` | `false` | `true` | The default counts only uncached prompt tokens, so a replica busy generating long answers looks idle. With `true`, each request also counts its expected output (its `max_tokens`, capped at 1,000) until it finishes. |
| `prefix-cache-affinity-filter` `peakPrefillThroughput` | `15928` | `1000` | Tokens per second used to turn in-flight tokens into a wait. |
| `prefix-cache-affinity-filter` `maxTTFTPenaltyMs` | `18000` | `1500` | At 1,000 tokens/s, the router leaves the sticky replica once it has 1,500 more in-flight tokens than another replica. |

llm-d's default `peakPrefillThroughput` of 15,928 tokens/s is calibrated for
Qwen 32B on two H100 80GB GPUs with vLLM 0.19 (see the
[affinity filter's README](https://github.com/llm-d/llm-d-router/blob/v1.0.0/pkg/epp/framework/plugins/scheduling/filter/prefixcacheaffinity/README.md#config)
in llm-d-router v1.0.0). On this CPU, "Where the first token comes from"
measured a prefill rate of about 800 to 900 tokens/s in our runs, so this
recipe uses a round 1,000.

This example also shows one of the limits below. The router does not know
that B can load the conversation from Valkey: it scores B as if B had to
recompute every token. Here A was busy enough that B still won, but in a closer
case the router underrates a neighbor that Valkey would make cheap.

Every placement so far relied on the router's index being correct when the
request arrived. The next section restarts the router and checks whether it
still is.

## The index outlives the router

So far the router's index has been in Valkey database 1. The router can also
keep it in its own memory: [`epp/precise-memory.yaml`](epp/precise-memory.yaml)
is `epp/precise-valkey.yaml` without the `redisConfig` block. While the router
runs, both make the same routing decisions. They differ when the router
restarts.

A restarted router with an in-memory index starts empty. When it connects to a
replica's event stream, it asks the replica to replay its recent events and
rebuilds what it can from them. vLLM's replay buffer keeps the last 10,000
event batches by default. This recipe shrinks it to 32 batches
(`KV_REPLAY_BUFFER_STEPS` in `docker-compose.yml`) so the boundary shows up
within seconds of traffic. How much real traffic 10,000 batches covers depends
on the load. A restarted router with the index in Valkey reads the index it
wrote before the restart.

The cell runs the same restart in both modes:

1. Switch the router to the mode, and warm four reviewers' conversations (one
   question each) through it.
2. Send 48 short prompts straight to each replica. Each fills one 64-token
   block, so each adds one event batch, and the warm conversations fall out of
   the 32-batch replay buffer. 48 blocks are well under the 128 a replica
   holds, so the conversations stay in the replicas' memory.
3. Restart the router (`docker compose restart epp`) and ask each reviewer's
   second question through it.

For each mode the table shows how many blocks the new router process
re-admitted from the replay, and the affinity filter's decision on the four
follow-ups: `sticky` means the router found the conversation on a replica,
`no_match` means it found nothing and placed the request by load alone. The
last columns show where the follow-ups' cached tokens came from. The cell took
about a minute in our runs.

![Two lanes, same events: warm four conversations, send more traffic, restart the router, ask a follow-up. In the lane with the index in router memory, the restart empties the index; the replay covers only the last 32 KV-event batches, which no longer include the warm conversations, so the follow-up is no_match and is placed by load alone, on either server; if it lands on the other one, that server reloads the prefix from Valkey. In the lane with the index in Valkey database 1, the restarted router reads the index, the follow-up is sticky, and it is a local hit on the warm server.](images/05-router-restart.svg)

In [ ]:
RESTART_CONVERSATIONS = 4  # even if all land on one replica: 4 x 18 blocks + filler < 128
FILLER_PER_REPLICA = 48  # more than the 32-batch replay buffer, far fewer than a replica's 128 blocks


def fill_replay_buffers(per_replica=FILLER_PER_REPLICA):
    """Short prompts straight to each replica: one full 64-token block, so one event batch, each."""
    for client in direct.values():
        for _ in range(per_replica):
            prompt = f"Filler note {uuid.uuid4().hex}. " + "The tenant keeps the premises clean. " * 10
            client.completions.create(model=MODEL, prompt=prompt, max_tokens=1, temperature=0)


def restart_trial(strategy):
    """Warm conversations, roll the replay buffers past them, restart the router, ask turn 2."""
    use_routing(strategy)
    conversations, warm_rows = run_conversations(router, strategy.split("-")[1],
                                                    count=RESTART_CONVERSATIONS, turns=1)
    time.sleep(2.5)  # let the router's 2 s speculative entries expire (Notes on the router index)
    fill_replay_buffers()
    time.sleep(1)    # the router applies the filler events asynchronously
    index_before = key_census(), index_by_replica()
    compose("restart", "epp")
    wait_for_router()
    time.sleep(1)    # let the replay finish
    replayed = metric(scrape(EPP_METRICS), "llm_d_epp_kv_cache_index_admissions_total")
    index_after = key_census(), index_by_replica()
    decisions_before = affinity_decisions()
    rows = []
    for i, messages in enumerate(conversations):
        messages.append({"role": "user", "content": QUESTIONS[1]})
        rows.append({"warm_on": warm_rows[i]["replica"], **ask(router, messages)})
    decisions = {k: int(v - decisions_before[k]) for k, v in affinity_decisions().items()}
    return {"replayed": replayed, "decisions": decisions, "rows": rows,
            "index_before": index_before, "index_after": index_after}


try:
    trials = {"in router memory": restart_trial("precise-memory"),
              "in Valkey (db 1)": restart_trial("precise-valkey")}
finally:
    use_routing("precise-valkey")  # the default for the rest of the notebook

print("index              re-admitted  follow-up decisions    local hits  local tokens  Valkey tokens")
for mode, trial in trials.items():
    rows, d = trial["rows"], trial["decisions"]
    local_hits = sum(r["local"] > 0 and r["valkey"] == 0 for r in rows)
    print(f"{mode:18s} {trial['replayed']:11.0f}  sticky {d['sticky']}, no_match {d['no_match']}"
          f"{'':5s} {local_hits:3d}/{len(rows)} {sum(r['local'] for r in rows):13d} "
          f"{sum(r['valkey'] for r in rows):14d}")
for mode, trial in trials.items():
    stayed = sum(r["replica"] == r["warm_on"] for r in trial["rows"])
    print(f"{mode}: {stayed}/{len(trial['rows'])} follow-ups landed on the replica that served turn 1")

(census_before, blocks_before), (census_after, blocks_after) = (
    trials["in Valkey (db 1)"]["index_before"], trials["in Valkey (db 1)"]["index_after"])
print("\nindex in Valkey, before and after the restart:")
for kind in ("router <hash>", "router engine:<hash>"):
    print(f"  {kind:22s} before {census_before[kind]:5d}   after {census_after[kind]:5d}")
print(f"  blocks per replica     before {blocks_before}   after {blocks_after}")

With the index in router memory, the replay re-admitted only the filler blocks
(64 in every run: 32 batches from each replica). The new router had forgotten
the conversations, so every follow-up was `no_match`. With both replicas idle,
placing by load is a coin flip: in our two runs of this cell, one and then all
four of the follow-ups landed on the warm replica by chance, and the rest went
to the other replica, which loaded the conversation from Valkey. Either way the
router no longer knew where the conversations were. With the index in Valkey,
the restarted router found every conversation (`sticky`) in every run, and all
the follow-ups were local hits. The block keys in database 1 and the blocks per
replica were the same before and after the restart. (The `engine:<hash>` count
can drop a little; "Notes on the router index" explains why.)

What the Valkey index adds is durability. While the router runs, both modes
route the same way. With the index in Valkey, placement survives a router
restart, and a router that starts later can read the same index. That is the
case for it here, not lower latency. Sharing one index between several routers
also needs shared in-flight load tracking, which llm-d v1.0.0 does not have (see
Limits). The replay buffer, speculative entries and stale entries after a router
outage are covered in "Notes on the router index".

The router is back on `precise-valkey` for the last section, the analyst's
batch.

## Batch jobs share the same servers

The analyst now runs a batch of questions about one lease. The batch must not
crowd out reviewers, and the batch jobs should get the same prefix routing as
chat, so that jobs about the same lease land where that lease is already
cached.

llm-d-async handles both with Valkey. It pulls jobs from a Valkey sorted set,
checks each job against a per-user quota kept in Valkey, sends admitted jobs
through the router, and pushes results to a Valkey list. The quota limits how
many of one user's jobs are admitted per time window, so a batch cannot take
every slot at once. Everything it needs to know is in
[`async/transport-config.json`](async/transport-config.json):

```json
"url": "redis://valkey:6379",
"queues": [{
  "queue_name": "request-sortedset",   "result_queue_name": "result-list",
  "igw_base_url": "http://envoy:8081", "request_path_url": "/v1/completions",
  "gate_type": "redis-quota",
  "gate_params": {"address": "valkey:6379", "attribute": "userid",
                  "mode": "rate-limit", "limit": 3, "window": "20s", "prefix": "quota:"}
}]
```

`igw_base_url` points at Envoy, so batch jobs go through the same router as
interactive traffic and land on whichever replica has their prefix cached.

![Batch path. Python adds jobs to the Valkey sorted set request-sortedset with ZADD. llm-d-async pulls them and checks the user's quota sorted set (quota:userid:...) in Valkey (3 jobs per user per 20 seconds). Admitted jobs go through Envoy and the router to a vLLM replica, and each answer is pushed to the Valkey list result-list. Held jobs stay queued until the quota window frees up.](images/06-batch-path.svg)

### Helper: how jobs are enqueued

Jobs normally come from llm-d-async's Go producer library. Here Python submits
them by doing exactly what that producer does: build the same JSON envelope
and run the same Lua script, which increments a per-deadline sequence number,
records the job's active token, and adds the job to the sorted set with a score
of `deadline + seq / 2^21` (earliest deadline first, then submission order).
The script is copied from llm-d-async v1.0.0
(`producer/redis_sortedset_producer.go`), licensed under the
[Apache License 2.0](https://github.com/llm-d/llm-d-async/blob/main/LICENSE).
It lives in [`helpers.py`](helpers.py) with `job_envelope()` and `submit_job()`,
so the recipe's tests submit jobs with the same code. The next cell prints the
script and the envelope builder.

In [ ]:
import inspect

import helpers

QUEUE, RESULTS = "request-sortedset", "result-list"

print(helpers.SUBMIT_REQUEST_LUA.strip(), end="\n\n")
print(inspect.getsource(helpers.job_envelope))


def gate_decisions():
    text = scrape(ASYNC_METRICS)
    return {"dispatched": metric(text, "llm_d_async_async_dispatched_requests_total"),
            "quota_exhausted": metric(text, "llm_d_async_async_gate_decisions_total",
                                      reason="quota_exhausted")}


print("submit_job() ready")

The analyst, `analyst-<id>`, submits five questions about the same lease. The
quota allows three jobs per user per 20 seconds, so three run straight away and
two wait in the sorted set until the window moves on. The cell polls Valkey once
a second and prints the queue, the quota key, and the results as they arrive.

In [ ]:
run_id = uuid.uuid4().hex[:8]
user = f"analyst-{run_id}"
quota_key = f"quota:userid:{user}"
batch_document = document(f"batch-{run_id}", clauses=26)
job_ids = [f"job-{run_id}-{i}" for i in range(5)]

counters_before, gates_before = replica_counters(), gate_decisions()
submitted_at = time.monotonic()
for i, job_id in enumerate(job_ids):
    submit_job(valkey, job_id, f"{batch_document}\n\nQuestion {i}: summarise clause {i * 5}.\nAnswer:",
               model=MODEL, userid=user, queue=QUEUE, results=RESULTS)

rows = valkey.custom_command(["ZRANGE", QUEUE, "0", "-1", "WITHSCORES"])
print(f"queued: {len(rows)} jobs in {QUEUE} (sorted set)")
member, score = next(iter(rows.items()))
print(f"one member (score {score}):\n  {member.decode()[:240]}...")

results = {}
print("\n  t   queued  quota count  quota TTL  results")
while len(results) < len(job_ids) and time.monotonic() - submitted_at < 90:
    # Take only this run's results; anything else in the list stays for its owner.
    for raw in valkey.custom_command(["LRANGE", RESULTS, "0", "-1"]) or []:
        result = json.loads(raw)
        if result["id"] in job_ids:
            results[result["id"]] = result
            valkey.custom_command(["LREM", RESULTS, "1", raw])
    print(f"{time.monotonic() - submitted_at:4.0f}s {valkey.zcard(QUEUE):6d} "
          f"{valkey.zcard(quota_key):12d} {valkey.ttl(quota_key):10d} {len(results):8d}")
    time.sleep(1)

counters_after, gates_after = replica_counters(), gate_decisions()
print("\njob     status  cached  answer")
reused = 0
for job_id in job_ids:
    result = results.get(job_id)
    if result is None:
        print(f"{job_id[-1]:>3s}     missing")
        continue
    body = json.loads(result["payload"])
    usage = body.get("usage", {})
    cached = (usage.get("prompt_tokens_details") or {}).get("cached_tokens", 0)
    reused += cached > 0
    answer = body["choices"][0]["text"].strip().replace("\n", " ")[:50] if "choices" in body else body
    print(f"{job_id[-1]:>3s}  {result.get('status_code', result.get('error_code'))!s:>8s}  {cached:6d}  {answer}")
print("\nserved through the router by:",
      {n: int(counters_after[n]["served"] - counters_before[n]["served"]) for n in REPLICA_URLS})
print("gate decisions:", {k: int(gates_after[k] - gates_before[k]) for k in gates_after})
local, from_valkey = (sum(int(counters_after[n][k] - counters_before[n][k]) for n in REPLICA_URLS)
                      for k in ("local", "valkey"))
print(f"jobs that reused the lease's KV (cached > 0): {reused}/{len(job_ids)} "
      f"({local} tokens from the replicas' memory, {from_valkey} from Valkey)")

The first three jobs were admitted at once. The quota is a sorted set per user
(`quota:userid:<user>`) with one member per admitted job, scored by admission
time, and an expiry of twice the window. With three members inside the 20-second
window, the other two jobs stayed in `request-sortedset` until the first
admissions aged out. `quota_exhausted` counts refusals, and one job can be
refused many times. On every poll (every 500 ms here) the processor dequeues
each waiting job, the gate refuses it again, and each refusal counts once
(`async_gate_decisions_total` in llm-d-async v1.0.0 `pkg/metrics/metrics.go`).
Two jobs held for about 20 seconds come to about 2 × 40 refusals, so expect the
counter to end up around 80 (80 to 82 in our runs).

The last line ties the batch back to routing. Jobs that reused the lease's KV
show it in the `cached` column: the router routes batch jobs by prefix exactly
like chat requests. In our runs 3 or 4 of the five jobs had cached tokens,
all from the replicas' own memory. The three jobs admitted together reach the
replicas before the lease is cached, so at least one of them computes it; jobs
that start after that, including the two the quota held back, find it cached.

The queue keys are now part of the census alongside the router index and the
LMCache chunks:

In [ ]:
print("batch queue keys:", sorted(k for k in scan_keys(valkey)
                                  if k in QUEUE_KEYS or k.startswith(QUEUE_PREFIXES))[:8], "...")
show_census("Valkey now")

## What you learned

### Why Valkey, role by role

| Role | Without Valkey | With Valkey | Evidence here |
| --- | --- | --- | --- |
| 🟦 Routing index | Lost when the router restarts; rebuilt only from the replicas' replay buffers | Survives a restart; a router that starts later reads it | "The index outlives the router" |
| 🟩 KV cache | A replica without the prefix in memory recomputes it | The other replica loads it from Valkey, several times faster than a recompute | "Where the first token comes from", "Baseline: random routing", "When the best server is busy" |
| 🟧 Queue and quota | One processor's in-memory state | Jobs, results and per-user admissions in Valkey, outside the processor | "Batch jobs share the same servers" |

- **Routing avoids work; the shared cache makes the misses cheap.** The router
  sent follow-ups back to the replica that had the conversation in memory.
  Random routing sent some elsewhere, where LMCache loaded them from Valkey
  instead of recomputing them.
- **A busy server hands off cheaply.** When the replica holding a conversation
  was full, the router moved the conversation, and the other replica loaded
  most of it from Valkey.
- **The index in Valkey survives the router.** After a restart that left an
  in-memory index with nothing to match, the router with its index in Valkey
  still sent every follow-up to the replica that held it.
- **Batch jobs share the same path.** Jobs queued in a Valkey sorted set and
  limited per user by a Valkey quota went through the same router as chat.

## Limits

For a production decision, this means that today the router index needs one
standalone Valkey node, LMCache needs a Valkey without a password until
[LMCache/LMCache#5917](https://github.com/LMCache/LMCache/pull/5917) is
released, and llm-d runs one router.

- **The prefix-cache index connects to a single Valkey node, so it can't use a
  sharded cluster or IAM authentication yet.** In llm-d-router v1.0.0 the index
  (`pkg/kvcache/kvblock/redis.go`) builds one `redis.NewClient` from
  `redis.ParseURL(address)`: one standalone node, with a password or TLS only if
  they fit in the URL, and no cluster client or token-refresh hook. In practice the index needs a
  standalone Valkey node today, with any password (or TLS, through a `rediss://`
  URL) written into the URL. It can't use a clustered Valkey, or one that
  requires short-lived IAM-style tokens. (The batch
  processor is only partly limited. llm-d-async v1.0.0's queue transport
  accepts `mode: cluster` and `mode: sentinel`, but the `redis-quota` gate used
  here takes a single `address` and opens a standalone connection to it
  (`docs/transports.md`). The producer script this notebook copies,
  `submitRequestScript`, is also standalone-only, because its keys span several
  cluster slots. In cluster mode the Go producer enqueues in two steps instead.)
- **The router can't see what LMCache stores in Valkey, so it routes only on
  what each server holds in its own GPU or CPU memory.** The router can score other
  storage tiers (llm-d-router v1.0.0 `pkg/kvcache/backend.go` has weights for
  `cpu`, `storage`, `shared_storage` and `object_store`), but nothing sends it
  events for them: the KV connector vLLM 0.28 loads here, `LMCacheMPConnector`
  (from LMCache 0.5.4), returns no events from `take_events()`. You saw the
  consequence in the busy-server example: the router assumed the neighbor had
  to recompute the whole conversation, when most of it came from Valkey.
- **LMCache's Valkey login bug breaks password-protected Valkey, standalone or
  cluster, until its one-line fix
  ([LMCache/LMCache#5917](https://github.com/LMCache/LMCache/pull/5917)) is
  released.** LMCache 0.5.4's Valkey adapter accepts a `username` and
  `password`, but passes them to valkey-glide in the wrong order
  (`ServerCredentials(username, password)` in
  `lmcache/v1/storage_backend/valkey/worker_pool.py`, where glide expects the
  password first), so the login fails. This stack, like the KV caching recipe,
  runs Valkey without authentication.
- **llm-d runs one router, so several router copies can't share this index
  yet.** llm-d v1.0.0's
  [precise prefix-cache routing guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/precise-prefix-cache-routing)
  runs the router as a single replica, and names two things that running
  several routers side by side would need: shared in-flight load and a complete
  index when a router starts. An index in Valkey gives the second, because a
  router that starts later reads everything the index already holds. It does not
  give the first. The `token-load-scorer` and the affinity filter's load check
  use in-flight tokens that each router process counts for itself, so two
  routers would each see about half the load and move conversations at the
  wrong time. This recipe shows the index surviving a router restart, not two
  routers sharing it.

### Where to start

Adopt the roles in the order this recipe builds them. The shared KV cache comes
first: it needs only the LMCache server and Valkey, and it is what makes a
request on the "wrong" server cheap (the [KV caching recipe](../kv-caching/)
covers it on its own). Precise routing comes next. It needs vLLM's prefix
caching and KV events, a router block size equal to vLLM's, and router load
settings calibrated for your hardware; put its index in its own Valkey
database from the start. The batch queue and quotas come last. They need only
Valkey and the router's address, and their jobs get prefix routing once the
router is in place.

## Notes on the router index

Details that matter to whoever runs the router. The sections above point
here.

**Why the index has a database of its own.** When a replica restarts with an
empty cache, the router drops that replica's entries by `SCAN`ning its whole
database and reading every key that does not start with `engine:` as a hash
(`RedisIndex.Clear` in llm-d-router v1.0.0). In a database shared with
LMCache's string keys, that scan stops at the first LMCache key with a
`WRONGTYPE` error and leaves stale entries pointing at the restarted replica.
Selecting database 1 in the router's URL (`/1`) avoids it with configuration
only.

**The replay buffer.** When the router connects to a replica's event stream, it
asks that replica to replay its recent events (port 5559). vLLM keeps the last
`buffer_steps` event batches for this: 10,000 by default, 32 in this recipe
(`KV_REPLAY_BUFFER_STEPS` in `docker-compose.yml`). vLLM publishes one batch for
each engine step in which blocks were stored or evicted, however many blocks
that is, so a batch is not a fixed amount of traffic. The filler prompts in
"The index outlives the router" are 64 to 127 tokens long, one full block, so
each adds one batch, and 48 of them push everything earlier out of a 32-batch
buffer. Blocks cached before the replay window are invisible to a fresh
in-memory index, but stay in the Valkey index until vLLM evicts them. With the
index in Valkey, re-applying replayed events that are already in the index
leaves the block keys and the blocks per replica unchanged.

**Orphaned `engine:<hash>` keys.** Each replica hashes blocks differently, so
when both replicas hold the same block, two `engine:<hash>` keys point at one
block key. When one replica evicts the block, the router removes that replica's
field from the block key. It keeps that replica's `engine:<hash>` mapping,
because the block key is not empty yet, and after that the mapping is never
deleted. Replaying the evicting replica's events after a restart can delete
such leftovers, but only if the block key is empty at the moment the eviction
is re-applied, which depends on how the two replicas' replays interleave. So
across a restart the `engine:<hash>` count drops by some, all or none of them.

**Speculative entries.** With `speculativeIndexing: true` the router records a
routed request's blocks against the chosen replica for 2 seconds, before the
replica's events confirm them. The timer that removes them runs in the router
process, so a router that stops inside that window leaves them in Valkey. That
is why the restart cell waits 2.5 seconds before it restarts the router.

**Stale entries after a router outage.** llm-d-router v1.0.0 can only notice
that a replica restarted while the router is running. It compares each event's
sequence number with the last one it received from that replica, which it holds
in memory, and clears the replica's entries when the number drops
(`pkg/kvevents/zmq_subscriber.go`). A router that starts fresh has nothing to
compare against. If a replica restarts while the router is down, or evicts
blocks while the router is down and those events leave its replay buffer before
the router is back, the new router never removes those entries, and the Valkey
index keeps pointing at blocks the replica no longer has. A fresh in-memory
index would not contain them. The in-memory run in "The index outlives the
router" is a small case of this: while that router ran, nothing updated
database 1, so blocks the replicas evicted in the meantime can still be listed
there. In this stack the cost is limited. LMCache still has that KV in Valkey,
so a request routed on a stale entry misses the expected local hit but loads
the prefix from Valkey instead of recomputing it.

## Configuration reference

The settings this recipe sets on purpose. Everything else is the upstream
default.

| File | Setting | Value | Why |
| --- | --- | --- | --- |
| `epp/precise-valkey.yaml` | `redisConfig.address` | `redis://valkey:6379/1` | Router index in Valkey, in a database of its own |
| `epp/precise-valkey.yaml` | `blockSizeTokens` | `64` | Must equal vLLM `--block-size` |
| `epp/precise-valkey.yaml` | `addEstimatedOutputTokens` | `true` | Count decode work as load |
| `epp/precise-valkey.yaml` | `peakPrefillThroughput` | `1000` | CPU prefill rate (tokens/s) |
| `epp/precise-valkey.yaml` | `maxTTFTPenaltyMs` | `1500` | Leave the sticky replica at 1,500 more in-flight tokens |
| `epp/precise-memory.yaml` | no `redisConfig` | | Same router with the index in its own memory ("The index outlives the router") |
| `epp/endpoints.yaml` | `address` | `172.16.89.11`, `.12` | Must equal each replica's `POD_IP` and compose address |
| `docker-compose.yml` (EPP) | `EPP_STRATEGY` | `precise-valkey` (default), `precise-memory`, `random` | Which router config in `epp/` the EPP loads |
| `docker-compose.yml` (vLLM) | `--enable-prefix-caching --block-size 64` | | Local prefix cache the router routes towards |
| `docker-compose.yml` (vLLM) | `--kv-events-config` | ZMQ on 5556, replay on 5559, topic `kv@<POD_IP>:8000@<model>` | Feeds the router's index |
| `docker-compose.yml` (vLLM) | `KV_REPLAY_BUFFER_STEPS` (`buffer_steps`) | `32` (vLLM default 10,000) | Short replay window, so the in-memory index loses old conversations within seconds |
| `docker-compose.yml` (vLLM) | `--kv-cache-memory-bytes 96M` | 128 blocks of 64 tokens | Room for this notebook's conversations on each replica |
| `docker-compose.yml` (vLLM) | `--max-num-seqs 2` | | Two long jobs fill a replica (busy-server example) |
| `docker-compose.yml` (LMCache) | `--l2-adapter` | `{"type":"valkey","startup_nodes":"valkey:6379"}` | KV chunks in Valkey |
| `async/transport-config.json` | `url`, `gate_params.address` | `redis://valkey:6379`, `valkey:6379` | Queue and quota in Valkey |
| `async/transport-config.json` | `igw_base_url` | `http://envoy:8081` | Batch jobs go through the router |
| `async/transport-config.json` | `limit`, `window` | `3`, `20s` | Per-user quota |

## Cleanup

```bash
docker compose down
```

This removes the containers, and with them everything in Valkey (it runs
without persistence). The `hf-cache` volume with the model weights is kept, so
the next `docker compose up` skips the download; `docker compose down -v`
removes it too.

## Next steps

- **Add a third replica.** A suggestion this recipe has not tested: add a
  `vllm-c` service to `docker-compose.yml` (a copy of `vllm-b` with the next
  fixed address, for example `172.16.89.13`, as both its `ipv4_address` and its
  `POD_IP`, and its own localhost port), add it to `epp/endpoints.yaml`, add
  its URL to `REPLICA_URLS` in the setup cell, and re-run "Routing: follow-ups
  find their server", "Baseline: random routing" and "When the best server is
  busy". The router subscribes to the new replica's events, and the index and
  KV cache are shared with it. Each vLLM replica peaked at about 3.7 to 3.8 GiB in
  our arm64 runs, so give the container runtime that much more memory.
- **Deploy on Kubernetes.** llm-d's
  [precise prefix-cache routing guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/precise-prefix-cache-routing)
  covers the Kubernetes deployment. It runs one router replica: two EPP
  instances pointed at the same Valkey database would share one index, but
  each would track only its own in-flight load.
- **Production batch.** llm-d's
  [batch serving guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/batch-serving)
  covers llm-d-async on Kubernetes, and llm-d-async's
  [transport docs](https://github.com/llm-d/llm-d-async/blob/v1.0.0/docs/transports.md)
  describe its Valkey cluster and sentinel modes for the queue (the quota gates
  still connect to one node).
- **Size the KV cache.** The [KV caching recipe](../kv-caching/) covers LMCache's
  Valkey options, including TTLs to bound how much KV Valkey keeps.